# Dance Library - Thumbnail Generator & Repair Tool

Generates and repairs thumbnails for videos in the dance library.

**Modes:**
- **Dry run** (`DRY_RUN = True`) — scans for issues and reports what *would* be fixed, without changing anything
- **Live run** (`DRY_RUN = False`) — actually generates, uploads, and updates DB records

**What it detects:**
1. **MISSING** — DB record has no `thumbnail_path` at all
2. **BROKEN** — DB record has a `thumbnail_path` but the file doesn't exist in R2
3. **OK** — thumbnail exists and is valid (skipped)

**How it works:**
1. Fetches all media records from Supabase
2. Lists actual thumbnails in R2 to verify which ones really exist
3. Matches records needing repair to local video files by `original_filename`
4. Uses ffmpeg to extract a frame at ~1 second — same parameters as the website (640px wide, WebP @ 80% quality)
5. Uploads the thumbnail to R2 and updates the database record

**Dependencies:** `pip install imageio-ffmpeg supabase boto3 python-dotenv tabulate`
(`imageio-ffmpeg` bundles a local ffmpeg binary — no global install needed)

In [11]:
# Install dependencies (run once)
# !pip install imageio-ffmpeg supabase boto3 python-dotenv tabulate

In [ ]:
import os
from dotenv import load_dotenv

# ══════════════════════════════════════════════
# DRY RUN — set to False to actually upload/update
# ══════════════════════════════════════════════
DRY_RUN = True

load_dotenv()

def clean_env(key):
    """Get env var and strip any inline # comments that dotenv may include."""
    val = os.getenv(key, "")
    return val.split("#")[0].strip() if val else ""

# ── Supabase config ──
SUPABASE_URL = clean_env("VITE_SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = clean_env("VITE_SUPABASE_SERVICE_ROLE_KEY")

# ── R2 config ──
R2_ACCOUNT_ID = clean_env("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = clean_env("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = clean_env("R2_SECRET_ACCESS_KEY")
R2_BUCKET_NAME = clean_env("R2_BUCKET_NAME")

# ── Local video folders ──
VIDEO_FOLDERS = [
    r"C:\Users\minds\Videos\DANCE\Bachata Moves",
    r"C:\Users\minds\Videos\DANCE\Kizomba Moves",
    r"C:\Users\minds\Videos\DANCE\Salsa Moves",
    r"C:\Users\minds\Videos\DANCE\Tango Moves",
    r"C:\Users\minds\Videos\DANCE\on1 Moves",
    r"C:\Users\minds\Videos\DANCE\on2 Moves",
]

# Validate
missing = [k for k, v in {
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY,
    "R2_ACCOUNT_ID": R2_ACCOUNT_ID,
    "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
    "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
    "R2_BUCKET_NAME": R2_BUCKET_NAME,
}.items() if not v]

if missing:
    print(f"Missing config: {', '.join(missing)}")
    print("Set them above or create a .env file in the TESTING folder.")
else:
    mode = "DRY RUN (no changes will be made)" if DRY_RUN else "LIVE RUN (will upload & update DB)"
    print(f"All config loaded. Mode: {mode}")

All config loaded. Mode: LIVE RUN (will upload & update DB)


In [13]:
import boto3
from supabase import create_client

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
print("Supabase connected.")

r2 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)
print("R2 connected.")

Supabase connected.
R2 connected.


In [14]:
# ── Step 1: Fetch ALL media records ──
all_records = []
page_size = 1000
offset = 0

while True:
    res = sb.table("media").select(
        "id, storage_path, original_filename, thumbnail_path"
    ).range(offset, offset + page_size - 1).execute()
    all_records.extend(res.data)
    if len(res.data) < page_size:
        break
    offset += page_size

print(f"Found {len(all_records)} total media records in database.")

Found 483 total media records in database.


In [15]:
# ── Step 2: Check which thumbnails actually exist in R2 ──
from tabulate import tabulate

r2_thumb_keys = set()
continuation_token = None

while True:
    kwargs = {"Bucket": R2_BUCKET_NAME, "MaxKeys": 1000, "Prefix": "thumbs/"}
    if continuation_token:
        kwargs["ContinuationToken"] = continuation_token
    response = r2.list_objects_v2(**kwargs)
    for obj in response.get("Contents", []):
        r2_thumb_keys.add(obj["Key"])
    if response.get("IsTruncated"):
        continuation_token = response["NextContinuationToken"]
    else:
        break

print(f"Found {len(r2_thumb_keys)} thumbnails in R2.\n")

# ── Categorize each record ──
ok_records = []
missing_records = []   # thumbnail_path is NULL
broken_records = []    # thumbnail_path set but not in R2

for r in all_records:
    tp = r.get("thumbnail_path")
    if not tp:
        missing_records.append(r)
    elif tp not in r2_thumb_keys:
        broken_records.append(r)
    else:
        ok_records.append(r)

print(f"OK (thumbnail exists):     {len(ok_records)}")
print(f"MISSING (no thumbnail):    {len(missing_records)}")
print(f"BROKEN (ref but no file):  {len(broken_records)}")

# ── Show details for records needing work ──
needs_work = missing_records + broken_records

if missing_records:
    print(f"\nMISSING records:")
    rows = [[r["id"][:8] + "...", r.get("original_filename", "N/A")[:50]] for r in missing_records]
    print(tabulate(rows, headers=["ID", "Filename"], tablefmt="simple"))

if broken_records:
    print(f"\nBROKEN records:")
    rows = [[r["id"][:8] + "...", r.get("original_filename", "N/A")[:50], r.get("thumbnail_path", "")] for r in broken_records]
    print(tabulate(rows, headers=["ID", "Filename", "Broken Path"], tablefmt="simple"))

if not needs_work:
    print("\nAll records have valid thumbnails. Nothing to do.")

Found 0 thumbnails in R2.

OK (thumbnail exists):     0
MISSING (no thumbnail):    483
BROKEN (ref but no file):  0

MISSING records:
ID           Filename
-----------  --------------------------------
a7074597...  IMG_6589.MP4
8e821c8c...  IMG_6614.MP4
dfe69b32...  IMG_6228.MOV
f93e8710...  IMG_7901.MOV
1f8902d4...  IMG_7902.MOV
93e49d4b...  IMG_7900.MOV
0b93d47c...  IMG_7933.MP4
e44a72ce...  IMG_7908.MP4
44bf0b53...  IMG_8016.MP4
54d2a2a4...  IMG_8252.MOV
33f259a6...  IMG_8254.MOV
e1439b7d...  IMG_8253.MOV
cf1d1a9c...  IMG_8257.MOV
32d059c7...  IMG_0031.MOV
c31ce845...  2024_06_02_20_44_34_IMG_0462.MOV
36484ecb...  2024_07_01_21_53_50_IMG_0847.MP4
3d3d7b48...  IMG_0010.MOV
bb8b321d...  IMG_0188.MOV
091ebbd2...  IMG_0209.MP4
efca9f8e...  IMG_0187.MOV
09037a1e...  IMG_1393.MOV
e9a0f1a5...  IMG_0210.MP4
2ccc4777...  2024_07_04_10_14_14_IMG_0911.MP4
57bc1380...  IMG_0211.MP4
560f3d58...  2024_07_09_19_54_25_IMG_0975.MOV
8696804c...  IMG_0212.MP4
0e46faa6...  2024_06_02_22_49_11_IMG_0463.

In [16]:
# ── Step 3: Scan local video folders ──
VIDEO_EXTENSIONS = {".mov", ".mp4", ".m4v", ".avi", ".mkv", ".webm"}

local_files = {}  # filename (case-insensitive) -> full path

for folder in VIDEO_FOLDERS:
    if not os.path.isdir(folder):
        print(f"  Skipping (not found): {folder}")
        continue
    for f in os.listdir(folder):
        ext = os.path.splitext(f)[1].lower()
        if ext in VIDEO_EXTENSIONS:
            local_files[f.lower()] = os.path.join(folder, f)

print(f"Found {len(local_files)} local video files across {len(VIDEO_FOLDERS)} folders.")

Found 478 local video files across 6 folders.


In [17]:
# ── Step 4: Match records needing work to local files ──
matched = []
unmatched = []

for r in needs_work:
    fname = (r.get("original_filename") or "").lower()
    if fname in local_files:
        status = "MISSING" if r in missing_records else "BROKEN"
        matched.append({**r, "local_path": local_files[fname], "issue": status})
    else:
        unmatched.append(r)

print(f"Matched to local files: {len(matched)}")
print(f"Unmatched (no local file): {len(unmatched)}")

if unmatched:
    print("\nCan't fix (no local video found):")
    for r in unmatched[:20]:
        print(f"  {r.get('original_filename', 'N/A')}")

if not matched:
    print("\nNothing to generate.")

Matched to local files: 478
Unmatched (no local file): 5

Can't fix (no local video found):
  IMG_0266.HEIC
  IMG_9385.HEIC
  IMG_9386.HEIC
  IMG_9186.HEIC
  IMG_9244.HEIC


In [18]:
import subprocess
import tempfile
import imageio_ffmpeg

# Get the local ffmpeg binary bundled with imageio-ffmpeg
FFMPEG_PATH = imageio_ffmpeg.get_ffmpeg_exe()
print(f"Using ffmpeg: {FFMPEG_PATH}")

# ── Same parameters as the website (WEBSITE/src/lib/ffmpeg.ts) ──
TARGET_WIDTH = 640
WEBP_QUALITY = 80
SEEK_TIME_SEC = 1.0

def extract_uuid_from_path(storage_path: str) -> str:
    """Extract UUID from storage_path like 'videos/abc-def.MOV'"""
    basename = storage_path.split("/")[-1]
    return os.path.splitext(basename)[0]

def generate_thumbnail(video_path: str) -> bytes | None:
    """Extract a frame at ~1s, scale to 640px wide, encode as WebP @ 80% quality.
    Matches the website's canvas-based thumbnail generation."""
    with tempfile.NamedTemporaryFile(suffix=".webp", delete=False) as tmp:
        tmp_path = tmp.name

    try:
        result = subprocess.run(
            [
                FFMPEG_PATH,
                "-ss", str(SEEK_TIME_SEC),   # seek to 1 second
                "-i", video_path,
                "-vframes", "1",              # single frame
                "-vf", f"scale={TARGET_WIDTH}:-2",  # 640px wide, even height
                "-quality", str(WEBP_QUALITY),
                "-y",                         # overwrite
                tmp_path,
            ],
            capture_output=True,
            timeout=30,
        )

        if result.returncode != 0:
            # If seeking to 1s fails (video shorter than 1s), retry at frame 0
            result = subprocess.run(
                [
                    FFMPEG_PATH,
                    "-i", video_path,
                    "-vframes", "1",
                    "-vf", f"scale={TARGET_WIDTH}:-2",
                    "-quality", str(WEBP_QUALITY),
                    "-y",
                    tmp_path,
                ],
                capture_output=True,
                timeout=30,
            )

        if result.returncode != 0 or not os.path.exists(tmp_path):
            return None

        with open(tmp_path, "rb") as f:
            return f.read()
    except (subprocess.TimeoutExpired, OSError):
        return None
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)

mode_label = "DRY RUN" if DRY_RUN else "LIVE"
print(f"[{mode_label}] Ready to process {len(matched)} thumbnails.")
print(f"Target: {TARGET_WIDTH}px wide, WebP quality {WEBP_QUALITY}")

Using ffmpeg: c:\Users\minds\miniforge3\envs\Primary\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe
[LIVE] Ready to process 478 thumbnails.
Target: 640px wide, WebP quality 80


In [19]:
# ── Step 6: Generate, upload, and update DB ──
results = {"success": 0, "gen_failed": 0, "upload_failed": 0, "db_failed": 0}
errors = []

for i, item in enumerate(matched):
    uuid = extract_uuid_from_path(item["storage_path"])
    thumb_key = f"thumbs/{uuid}.webp"
    fname = item.get("original_filename", "unknown")
    issue = item.get("issue", "MISSING")

    print(f"[{i+1}/{len(matched)}] [{issue}] {fname} ... ", end="", flush=True)

    # Generate thumbnail
    thumb_bytes = generate_thumbnail(item["local_path"])
    if thumb_bytes is None:
        print("FAILED (generation)")
        results["gen_failed"] += 1
        errors.append((fname, "thumbnail generation failed"))
        continue

    size_kb = len(thumb_bytes) / 1024

    if DRY_RUN:
        print(f"WOULD upload ({size_kb:.1f} KB) -> {thumb_key}")
        results["success"] += 1
        continue

    # Upload to R2
    try:
        r2.put_object(
            Bucket=R2_BUCKET_NAME,
            Key=thumb_key,
            Body=thumb_bytes,
            ContentType="image/webp",
        )
    except Exception as e:
        print(f"FAILED (upload: {e})")
        results["upload_failed"] += 1
        errors.append((fname, f"R2 upload failed: {e}"))
        continue

    # Update DB (only needed for MISSING; BROKEN already has the correct path)
    if issue == "MISSING":
        try:
            sb.table("media").update({"thumbnail_path": thumb_key}).eq("id", item["id"]).execute()
        except Exception as e:
            print(f"FAILED (db update: {e})")
            results["db_failed"] += 1
            errors.append((fname, f"DB update failed: {e}"))
            continue

    print(f"OK ({size_kb:.1f} KB) -> {thumb_key}")
    results["success"] += 1

[1/478] [MISSING] IMG_6589.MP4 ... OK (44.5 KB) -> thumbs/0d2472e5-f9f7-4a25-929a-b1ec2d749ac6.webp
[2/478] [MISSING] IMG_6614.MP4 ... OK (43.3 KB) -> thumbs/930b3309-e3cc-413f-bc31-e12a35afc73d.webp
[3/478] [MISSING] IMG_6228.MOV ... OK (47.0 KB) -> thumbs/8c4ec416-fab8-4184-8d0f-0feac5c8ae67.webp
[4/478] [MISSING] IMG_7901.MOV ... OK (61.6 KB) -> thumbs/71663dcb-92d4-4725-bdb7-c2b77593e327.webp
[5/478] [MISSING] IMG_7902.MOV ... OK (52.9 KB) -> thumbs/a0211936-da00-4811-80ea-b7e346e3d1b0.webp
[6/478] [MISSING] IMG_7900.MOV ... OK (66.4 KB) -> thumbs/04e0d7b7-6c05-4239-b1a5-ead9a08d4e72.webp
[7/478] [MISSING] IMG_7933.MP4 ... OK (62.6 KB) -> thumbs/538e49d8-9efc-48fa-8d95-7fac9d1615c7.webp
[8/478] [MISSING] IMG_7908.MP4 ... OK (49.4 KB) -> thumbs/2854e11f-13a2-47c1-8ba1-20860eec7927.webp
[9/478] [MISSING] IMG_8016.MP4 ... OK (34.0 KB) -> thumbs/1f4e5869-7e4a-4568-ab99-ccc742ad029e.webp
[10/478] [MISSING] IMG_8252.MOV ... OK (50.3 KB) -> thumbs/01104de4-1bf5-4016-8c69-cf8dbbe35480.webp

In [20]:
# ── Summary ──
mode_label = "DRY RUN" if DRY_RUN else "LIVE"
action_word = "would generate" if DRY_RUN else "generated"

print("=" * 60)
print(f"SUMMARY ({mode_label})")
print("=" * 60)
print(f"")
print(f"Total DB records:          {len(all_records)}")
print(f"  OK (valid thumbnail):    {len(ok_records)}")
print(f"  MISSING (no thumbnail):  {len(missing_records)}")
print(f"  BROKEN (bad reference):  {len(broken_records)}")
print(f"")
print(f"Matched to local files:    {len(matched)}")
print(f"Unmatched (can't fix):     {len(unmatched)}")
print(f"")
print(f"Thumbnails {action_word}:  {results['success']}")
print(f"Generation failed:         {results['gen_failed']}")
if not DRY_RUN:
    print(f"Upload failed:             {results['upload_failed']}")
    print(f"DB update failed:          {results['db_failed']}")
print(f"")

if errors:
    print("Errors:")
    for fname, err in errors:
        print(f"  {fname}: {err}")
elif not needs_work:
    print("All records have valid thumbnails. Nothing to do!")
elif results['success'] == len(matched) and not unmatched:
    if DRY_RUN:
        print(f"All {results['success']} thumbnails can be generated successfully.")
        print("Set DRY_RUN = False and re-run to apply changes.")
    else:
        print("All thumbnails generated and uploaded successfully!")
else:
    if DRY_RUN:
        print("Review the results above, then set DRY_RUN = False to apply.")
    else:
        print("Done with some items skipped. Check errors/unmatched above.")

SUMMARY (LIVE)

Total DB records:          483
  OK (valid thumbnail):    0
  MISSING (no thumbnail):  483
  BROKEN (bad reference):  0

Matched to local files:    478
Unmatched (can't fix):     5

Thumbnails generated:  478
Generation failed:         0
Upload failed:             0
DB update failed:          0

Done with some items skipped. Check errors/unmatched above.
